<a href="https://colab.research.google.com/github/mide-etan/HackBio-StageZero/blob/main/Diabetes_Predictions_Using_Explainable_Machine_Learning_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Diabetes Predictions Using Explainable Machine Learning Models**
---


***About***

---


Diabetes mellitus, a chronic metabolic disorder affecting over 463 million adults globally, demands personalized treatment strategies to mitigate its rising prevalence and complications. This study harnesses advanced machine learning (ML) techniques to develop data-driven, individualized approaches for diabetes management using multimodal data from the ShanghaiT1DM, ShanghaiT2DM, and Open D1NAMO datasets. Employing a robust methodology, we integrate data preprocessing, feature engineering, patient phenotyping via K-Means and Hierarchical Clustering, time-series modeling with Long Short-Term Memory (LSTM) and Convolutional Neural Networks (CNNs), reinforcement learning through Contextual Bandits and Deep Q-Networks (DQNs), and supervised learning with Random Forest, XGBoost, and Deep Neural Networks (DNNs). These models predict clinical outcomes, optimize treatment plans, and forecast glucose dynamics over short-term horizons. To ensure clinical applicability, we incorporate explainable AI techniques, including SHAP and LIME, to provide transparent insights into model predictions. Our findings demonstrate the potential of ML to enhance glycemic control, reduce complications, and tailor interventions, paving the way for precision medicine in diabetes care and alleviating the global healthcare burden.

***Install Libraries.***

In [ ]:
#Core data handling and numerical computation
!pip install numpy pandas

# Data preprocessing and feature engineering
!pip install scikit-learn imbalanced-learn

# Machine learning models
!pip install xgboost

# Deep learning and neural networks
!pip install tensorflow keras

# Reinforcement learning
!pip install gym stable-baselines3
!pip install stable-baselines3[extra] --upgrade

# Model interpretation
!pip install shap lime

# Visualization (for evaluation and interpretation)
!pip install matplotlib seaborn

# Optional: for handling time-series data and clustering
!pip install tslearn

#Visualization
!pip install matplotlib seaborn plotly

# Evaluation metrics and statistical analysis
!pip install statsmodels scikit-posthocs

# Handling large datasets and performance optimization
!pip install dask
!pip install dask[dataframe]

# Optional: for advanced data manipulation and database integration
!pip install pyarrow sqlalchemy


***Import Neccessary Libraries***

In [ ]:
# Core data handling and numerical computation
import numpy as np
import pandas as pd
import dask.dataframe as dd
import pyarrow as pa
import sqlalchemy as sa

# Data preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestRegressor  # For imputation
from scipy import stats

# Machine learning models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_absolute_error, mean_squared_error

# Deep learning and neural networks
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, MaxPooling1D, Flatten, Dropout
import torch
import torch.nn as nn
import torch.optim as optim

# Reinforcement learning
import gym
from gym import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.torch_layers import MlpExtractor #Instead of stable_baselines3.common.policies import MlpPolicy
from stable_baselines3.common.buffers import ReplayBuffer

# Clustering and time-series analysis
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from sklearn.metrics import silhouette_score

# Model interpretation
import shap
import lime
import lime.lime_tabular

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Evaluation metrics and statistical analysis
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp
from sklearn.metrics import confusion_matrix, roc_curve

# Utility for large datasets and progress tracking
from dask.diagnostics import ProgressBar
from sqlalchemy import create_engine
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("All necessary libraries imported successfully!")

All necessary libraries imported successfully!


***Loading and Parsing of Datasets***

In [ ]:
# Import necessary libraries (assumes prior import script was run)
import numpy as np
import pandas as pd
import os
import tarfile
import zipfile
from google.colab import drive
from scipy import stats
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)  # Set force_remount=True if you really need to

# Define paths to datasets
BASE_PATH     = Path('/content/drive/My Drive/TechX./Colab Notebooks')
EXTRACT_PATH  = Path('/content/datasets')           # Everything will be unpacked here
EXTRACT_PATH.mkdir(exist_ok=True)

# ZIP files located on Drive
ZIP_D1NAMO    = BASE_PATH / 'diabetes_subset_ecg_data.zip'
ZIP_SHANGHAI  = BASE_PATH / 'data1.zip'

def extract_zip(zip_file: Path, dest: Path):
    """Unzip *zip_file* into *dest* if it hasn’t been extracted yet."""
    if not zip_file.exists():
        raise FileNotFoundError(f"{zip_file} does not exist in Drive.")
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(dest)
    print(f"Extracted {zip_file.name} → {dest}")

extract_zip(ZIP_D1NAMO,  EXTRACT_PATH)
extract_zip(ZIP_SHANGHAI, EXTRACT_PATH)

# Optional: visual sanity check of extraction result
print("\n--- Directory tree under /content/datasets/ ---")
for root, dirs, files in os.walk(EXTRACT_PATH):
    indent = '  ' * (root.replace(str(EXTRACT_PATH), '').count(os.sep))
    print(f"{indent}{Path(root).name}/")
    for f in files:
        print(f"{indent}  {f}")
print("------------------------------------------------\n")


def find_dirs(keyword_list):
    """
    Return a list of directories under EXTRACT_PATH whose names contain
    *all* substrings in keyword_list (case‑insensitive).
    """
    hits = []
    for p in EXTRACT_PATH.rglob('*'):
        if p.is_dir():
            name = p.name.lower()
            if all(k.lower() in name for k in keyword_list):
                hits.append(p)
    return hits

# D1NAMO (exact dir known)
d1namo_dirs = [p for p in EXTRACT_PATH.glob('diabetes_subset_ecg_data*') if p.is_dir()]

# Shanghai datasets (names vary; match with keywords)
t1dm_dirs   = find_dirs(['shanghai', 't1dm'])
t2dm_dirs   = find_dirs(['shanghai', 't2dm'])

print("Located D1NAMO dirs   :", d1namo_dirs)
print("Located Shanghai T1DM:", t1dm_dirs)
print("Located Shanghai T2DM:", t2dm_dirs)


def load_all_csv_parquet(dir_paths):
    dfs = []
    for path in dir_paths:
        for fp in path.rglob('*'):
            if fp.suffix == '.csv':
                dfs.append(pd.read_csv(fp))
            elif fp.suffix == '.parquet':
                dfs.append(pd.read_parquet(fp))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# Load each dataset group
d1namo_df      = load_all_csv_parquet(d1namo_dirs)
shanghai_t1dm_df = load_all_csv_parquet(t1dm_dirs)
shanghai_t2dm_df = load_all_csv_parquet(t2dm_dirs)

print("Shapes → D1NAMO:", d1namo_df.shape,
      "| T1DM:", shanghai_t1dm_df.shape,
      "| T2DM:", shanghai_t2dm_df.shape)


dataset = pd.concat([d1namo_df, shanghai_t1dm_df, shanghai_t2dm_df],
                    ignore_index=True)
print("Unified dataset shape:", dataset.shape)


def assess_data_quality(df, label):
    total_cells = df.shape[0] * df.shape[1]
    completeness = df.notnull().sum().sum() / total_cells
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if not numeric_cols.empty:
        z_scores = np.abs(stats.zscore(df[numeric_cols]))
        consistency = (z_scores < 3).sum().sum() / (df.shape[0] * len(numeric_cols))
    else:
        consistency = np.nan  # No numeric columns
    print(f"{label}: completeness={completeness:.4f} | consistency={consistency:.4f}")

assess_data_quality(d1namo_df,      "D1NAMO")
assess_data_quality(shanghai_t1dm_df, "Shanghai‑T1DM")
assess_data_quality(shanghai_t2dm_df, "Shanghai‑T2DM")
assess_data_quality(dataset,         "Unified")


def temporal_alignment(df,
                       glucose_col='glucose',
                       timestamp_col='timestamp',
                       id_col='patient_id'):
    """Align each patient’s glucose series to a mean 24‑h reference curve."""
    if not {glucose_col, timestamp_col, id_col}.issubset(df.columns):
        print("❗ Required columns missing; skipping alignment.")
        return df

    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    ref_profile = (df
                   .groupby(df[timestamp_col].dt.hour)[glucose_col]
                   .mean()
                   .to_numpy())

    aligned = []
    for pid, grp in df.groupby(id_col):
        g = grp[glucose_col].to_numpy()
        if g.size < ref_profile.size:
            continue
        # Brute‑force circular shift to minimise squared error
        best_shift = min(range(-g.size//2, g.size//2),
                         key=lambda s: np.sum((np.roll(g, s)[:ref_profile.size] - ref_profile) ** 2))
        grp[glucose_col] = np.roll(g, best_shift)
        aligned.append(grp)

    return pd.concat(aligned, ignore_index=True)

dataset_aligned = temporal_alignment(dataset)
print("Temporal alignment complete. Final shape:", dataset_aligned.shape)


OUTPUT_CSV = EXTRACT_PATH / 'unified_dataset.csv'
dataset_aligned.to_csv(OUTPUT_CSV, index=False)
print(f"Saved cleaned dataset → {OUTPUT_CSV}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'Path' is not defined